# Sesión 3 · Lo típico y lo variable
**Análisis Estadístico de la Información — Especialización en Big Data (UCompensar)**

Este cuaderno (*notebook*) acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/diapositivas/sesion-3-lo-tipico-y-lo-variable.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el cuaderno no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tingua Domicilios S.A.S.*, una empresa **ficticia**
   (sus datos se inventan con una semilla fija y no corresponden a personas ni a empresas reales),
   y les da a las gráficas los colores del curso.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tingua Domicilios S.A.S.» (empresa FICTICIA).

Genera datos/pedidos.csv: los 12 000 pedidos que la plataforma entregó entre el
1 de enero y el 30 de junio de 2026 en cinco ciudades. Todo se inventa con una
semilla fija: los datos no corresponden a personas ni a empresas reales.

Solo usa la biblioteca estándar y, del generador aleatorio, solo random(), que
es lo único que Python garantiza igual entre versiones. Por eso la tabla sale
idéntica en Colab y en cualquier computador.

El caso sirve para todo el curso: variables de todos los tipos (sesión 1),
asimetrías y algunos atípicos (medidas y datos atípicos), una población
completa de la que se pueden sacar muestras (muestreo) y un tiempo de entrega
que depende de la distancia, la lluvia y otras variables (regresión).
"""

import csv
import math
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 2026
N_PEDIDOS = 12_000
INICIO = date(2026, 1, 1)
DIAS = 181  # del 1 de enero al 30 de junio de 2026

COLUMNAS = ["id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos",
            "valor_pedido", "medio_pago", "estrato", "suscriptor", "distancia_km",
            "lluvia", "temperatura_c", "tiempo_entrega_min", "calificacion"]

# ciudad: (peso, temperatura media °C, desviación, prob. de lluvia, minutos extra, factor de distancia)
CIUDADES = {
    "Bogotá":       (0.40, 14.5, 2.2, 0.30, 5.0, 1.15),
    "Medellín":     (0.22, 22.5, 2.3, 0.28, 2.0, 1.00),
    "Cali":         (0.16, 25.0, 2.3, 0.22, 0.0, 1.00),
    "Barranquilla": (0.12, 29.0, 1.8, 0.12, -1.0, 0.95),
    "Bucaramanga":  (0.10, 23.5, 2.2, 0.20, -2.0, 0.90),
}
LLUVIA_MES = {1: 0.6, 2: 0.7, 3: 1.0, 4: 1.4, 5: 1.3, 6: 0.8}

# franja: (peso, ajuste de temperatura °C, minutos extra)
FRANJAS = {
    "Mañana":   (0.15, -1.5, 0.0),
    "Almuerzo": (0.35, 1.5, 4.0),
    "Tarde":    (0.20, 1.0, 0.0),
    "Noche":    (0.30, -2.0, 3.0),
}

# categoría: (productos extra en promedio, precio medio por producto en COP, minutos de preparación)
CATEGORIAS = {
    "Restaurante": (1.3, 27_000, 3.0),
    "Mercado":     (6.0, 8_500, 4.0),
    "Farmacia":    (0.9, 21_000, 0.0),
    "Mascotas":    (1.2, 36_000, 1.0),
    "Floristería": (0.2, 88_000, 2.0),
}
# Qué se pide en cada franja (mismo orden que CATEGORIAS)
CATEGORIA_POR_FRANJA = {
    "Mañana":   (0.30, 0.35, 0.20, 0.10, 0.05),
    "Almuerzo": (0.78, 0.08, 0.07, 0.04, 0.03),
    "Tarde":    (0.40, 0.25, 0.15, 0.12, 0.08),
    "Noche":    (0.70, 0.10, 0.12, 0.05, 0.03),
}
ESTRATOS = {1: 0.06, 2: 0.22, 3: 0.33, 4: 0.20, 5: 0.11, 6: 0.08}


class _Azar:
    """Lo que hace falta del azar, construido solo sobre random()."""

    def __init__(self, semilla):
        self.r = random.Random(semilla)

    def u(self):
        return self.r.random()

    def elegir(self, opciones, pesos):
        x, acum = self.u() * sum(pesos), 0.0
        for opcion, peso in zip(opciones, pesos):
            acum += peso
            if x < acum:
                return opcion
        return opciones[-1]

    def normal(self, media=0.0, desv=1.0):
        # Box-Muller
        u1, u2 = 1.0 - self.u(), self.u()
        return media + desv * math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

    def exponencial(self, media):
        return -media * math.log(1.0 - self.u())

    def poisson(self, lam):
        # Knuth: suficiente para medias pequeñas
        limite, k, p = math.exp(-lam), 0, 1.0
        while True:
            p *= self.u()
            if p <= limite:
                return k
            k += 1


def _redondear(x, decimales=0):
    factor = 10 ** decimales
    return math.floor(x * factor + 0.5) / factor


def _pedido(az, dias, pesos_dias):
    fecha = az.elegir(dias, pesos_dias)
    ciudad = az.elegir(list(CIUDADES), [v[0] for v in CIUDADES.values()])
    _, t_media, t_desv, p_lluvia, min_ciudad, f_dist = CIUDADES[ciudad]
    franja = az.elegir(list(FRANJAS), [v[0] for v in FRANJAS.values()])
    _, t_franja, min_franja = FRANJAS[franja]
    categoria = az.elegir(list(CATEGORIAS), CATEGORIA_POR_FRANJA[franja])
    extra, precio, min_prep = CATEGORIAS[categoria]
    estrato = az.elegir(list(ESTRATOS), list(ESTRATOS.values()))

    suscriptor = az.u() < 0.10 + 0.06 * (estrato - 1)
    p_efectivo = 0.45 - 0.07 * (estrato - 1)
    medio_pago = az.elegir(["Efectivo", "Billetera digital", "Tarjeta"],
                           [p_efectivo, 0.25, 1 - p_efectivo - 0.25])

    n = 1 + az.poisson(extra)
    if categoria in ("Mercado", "Restaurante") and az.u() < 0.003:
        n = 20 + int(az.u() * 26)  # pedido corporativo: pocos, pero existen
    valor = n * precio * math.exp(az.normal(0, 0.22)) * (1 + 0.07 * (estrato - 3))
    valor = max(3_000, _redondear(valor / 100) * 100)

    distancia = math.exp(az.normal(math.log(3.0), 0.5)) * f_dist
    distancia = min(14.0, max(0.4, _redondear(distancia, 1)))

    lluvia = az.u() < p_lluvia * LLUVIA_MES[fecha.month]
    temperatura = _redondear(az.normal(t_media + t_franja - 1.8 * lluvia, t_desv), 1)

    tiempo = (11 + 3.1 * distancia + 0.45 * min(n, 15) + min_prep + min_franja + min_ciudad
              + 7.5 * lluvia - 2.5 * suscriptor + az.normal(0, 4) + az.exponencial(3))
    if az.u() < 0.005:
        tiempo += 55 + 95 * az.u()  # pedidos que se demoraron muchísimo
    tiempo = max(6.0, _redondear(tiempo, 1))

    latente = 6.4 - 0.065 * tiempo + 0.2 * suscriptor + az.normal(0, 0.6)
    estrellas = int(min(5, max(1, math.floor(latente + 0.5))))
    califico = az.u() < (0.80 if tiempo > 55 else 0.58)

    return {
        "fecha": fecha, "franja": franja, "ciudad": ciudad, "categoria": categoria,
        "n_productos": n, "valor_pedido": int(valor), "medio_pago": medio_pago,
        "estrato": estrato, "suscriptor": "Sí" if suscriptor else "No",
        "distancia_km": distancia, "lluvia": "Sí" if lluvia else "No",
        "temperatura_c": temperatura, "tiempo_entrega_min": tiempo,
        "calificacion": estrellas if califico else "",
    }


def crear_datos(carpeta="datos"):
    """Escribe <carpeta>/pedidos.csv y devuelve su ruta."""
    az = _Azar(SEMILLA)
    dias = [INICIO + timedelta(days=d) for d in range(DIAS)]
    pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]  # viernes a domingo, más pedidos
    pedidos = [_pedido(az, dias, pesos_dias) for _ in range(N_PEDIDOS)]
    orden_franja = list(FRANJAS)
    pedidos.sort(key=lambda p: (p["fecha"], orden_franja.index(p["franja"])))

    ruta = Path(carpeta) / "pedidos.csv"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        for i, p in enumerate(pedidos, start=1):
            escritor.writerow({"id_pedido": f"P{i:05d}", **p, "fecha": p["fecha"].isoformat()})
    miles = f"{N_PEDIDOS:,}".replace(",", "\u202f")  # 12 000, con espacio fino
    print(f"Listo: {ruta} con {miles} pedidos y {len(COLUMNAS)} columnas")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras genera_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2; con pandas 3 algunas salidas cambian."""
    import pandas as pd

    if int(pd.__version__.split(".")[0]) >= 3:
        print(f"Aviso: este Colab trae pandas {pd.__version__} y las diapositivas usan pandas 2.2. "
              "Algunas salidas se verán un poco distintas (por ejemplo, el texto sale como str "
              "y no como object).")


_aviso_version()


"""Sesión 3: estilo de las gráficas del curso y verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con genera_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

Tolerancias: ±0,05 en minutos y kilómetros; la desviación se acepta con las dos fórmulas
(entre n − 1 o entre n) porque cada grupo pasa de 1000 pedidos.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()


def _mostrar(tabla):
    try:
        from IPython.display import display
        display(tabla)
    except ImportError:
        print(tabla.to_string())


def _original():
    """Los pedidos tal como salen del generador, o None (con aviso) si no están."""
    import pandas as pd

    try:
        return pd.read_csv("datos/pedidos.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/pedidos.csv: ejecuta primero la celda que crea los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None


def _sobrantes(sobran, otros, nombres):
    """Avisa de argumentos de más o con otro nombre. True si hubo alguno."""
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    if sobran:
        print(f"✗  Sobran valores: el verificador recibe solo {len(nombres)} "
              f"({', '.join(n + '=' for n in nombres)})")
        return True
    return False


_COLUMNAS = {"id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos", "valor_pedido",
             "medio_pago", "estrato", "suscriptor", "distancia_km", "lluvia", "temperatura_c",
             "tiempo_entrega_min", "calificacion"}


def _un_numero(x, n_columna=None):
    """(valor, None) si x es un número, o una serie o lista con un solo número; si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            return None, f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta un solo número"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple)):
        valores = list(np.ravel(np.asarray(x, dtype=object)))
        if n_columna is not None and len(valores) == n_columna:
            return None, "esa es la columna entera, un valor por pedido: falta resumirla"
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if all(isinstance(v, float) and math.isnan(v) for v in valores):
                return None, ("salió nan: si restaste dos series, pandas las alinea por su índice "
                              "y lo que no coincide queda nan; calcula cada valor como un número")
            pista = ""
            if isinstance(x, pd.Series) and set(map(str, x.index)) & _COLUMNAS:
                pista = ": ¿elegiste la columna?"   # p. ej., pedidos.mean(numeric_only=True)
            elif isinstance(x, pd.Series) and {"count", "mean", "std"} <= set(map(str, x.index)):
                pista = ": describe() da varias medidas; quédate solo con la que se pide"
            return None, f"tiene {len(valores)} valores; hace falta uno solo{pista}"
        x = valores[0]   # mode() o quantile([0.25]) devuelven una serie de un solo valor
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if x is pd.NA:
        return None, "salió <NA>: revisa el cálculo (¿la columna tiene datos?)"
    if callable(x):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis del final, ()?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except (TypeError, ValueError, OverflowError):
        return None, "no es un número real"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo (¿la columna tiene datos?)"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _serie(x):
    """La serie que entregó el estudiante, o None si no es una."""
    import pandas as pd

    if isinstance(x, pd.DataFrame) and x.shape[1] == 1:
        x = x.iloc[:, 0]
    if not isinstance(x, pd.Series):
        return None
    if isinstance(x.index, pd.MultiIndex) and x.index.nlevels == 1:  # pedidos[["col"]].groupby…
        x = x.set_axis(x.index.get_level_values(0))
    return x


def _cerca(a, b, tolerancia):
    return a is not None and abs(a - b) <= tolerancia


def revisar_centro(media=..., mediana=..., moda=..., *sobran, **otros):
    """Ejercicio 1: media, mediana y moda de n_productos."""
    import pandas as pd

    if _sobrantes(sobran, otros, ["media", "mediana", "moda"]):
        return
    original = _original()
    if original is None:
        return
    columna = original["n_productos"]
    n = len(columna)
    verdad = {"media": columna.mean(), "mediana": columna.median(), "moda": columna.mode()[0]}
    veces_moda = int((columna == verdad["moda"]).sum())
    # medias y medianas de las otras columnas numéricas: para detectar la columna equivocada
    otras = original.select_dtypes("number").drop(columns="n_productos")
    de_otras = list(otras.mean()) + list(otras.median())
    bien = 0

    for nombre, x in [("media", media), ("mediana", mediana), ("moda", moda)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        v, problema = _un_numero(x, n)
        if problema:
            print(f"✗  {nombre}: {problema}")
            continue
        if nombre == "media":
            if _cerca(v, verdad["media"], 0.0051):
                print("✓  media")
                bien += 1
            elif _cerca(v, verdad["media"], 0.051):
                print("✗  media: casi: perdiste precisión al redondear; deja al menos dos decimales")
            elif _cerca(v, verdad["mediana"], 1e-9) or _cerca(v, verdad["moda"], 1e-9):
                print("✗  media: ese es el valor del medio o el más frecuente, o la media sin sus "
                      "decimales (¿usaste int()?); la media suma todos los valores, divide entre "
                      "cuántos son y conserva los decimales")
            elif v != round(v) and any(_cerca(v, o, 1e-6) for o in de_otras):
                print("✗  media: ese valor es de otra columna: ¿elegiste n_productos?")
            else:
                print("✗  media: no cuadra: ¿la calculaste sobre n_productos, con todos los pedidos?")
        elif nombre == "mediana":
            if _cerca(v, verdad["mediana"], 1e-9):
                print("✓  mediana")
                bien += 1
            elif v != round(v) and _cerca(v, verdad["media"], 0.051):
                print("✗  mediana: eso parece la media; la mediana es el valor del medio de los "
                      "datos ordenados")
            elif v != round(v) and any(_cerca(v, o, 1e-6) for o in de_otras):
                print("✗  mediana: ese valor es de otra columna: ¿elegiste n_productos?")
            else:
                print("✗  mediana: no cuadra: es el valor que deja la mitad de los pedidos a cada lado")
        else:
            if _cerca(v, verdad["moda"], 1e-9):
                print("✓  moda")
                bien += 1
            elif _cerca(v, veces_moda, 1e-9):
                print("✗  moda: ese número dice cuántas veces aparece el valor más frecuente; la "
                      "moda es el valor, no su frecuencia")
            elif v != round(v) and _cerca(v, verdad["media"], 0.051):
                print("✗  moda: eso parece la media; la moda es el valor que más se repite")
            else:
                print("✗  moda: no cuadra: es el valor de n_productos que más se repite")

    if bien == 3:
        tabla = pd.DataFrame({"n_productos": [verdad["media"], verdad["mediana"], verdad["moda"]]},
                             index=["media", "mediana", "moda"])
        _mostrar(tabla.round(2))
        print("✓  Las tres cuadran. Responde abajo: ¿cuál representa mejor al pedido típico?")


def revisar_medianas(estrato=..., calificacion=..., *sobran, **otros):
    """Ejercicio 2: mediana del estrato y de la calificación (pedidos calificados), leídas en la
    acumulada."""
    import pandas as pd

    if _sobrantes(sobran, otros, ["estrato", "calificacion"]):
        return
    original = _original()
    if original is None:
        return
    n = len(original)
    resultado = {}
    bien = 0

    for nombre, x, columna, categorias in [
        ("estrato", estrato, "estrato", [1, 2, 3, 4, 5, 6]),
        ("calificacion", calificacion, "calificacion", [1, 2, 3, 4, 5]),
    ]:
        datos = original[columna].dropna()
        acumulada = datos.value_counts(normalize=True).reindex(categorias, fill_value=0).cumsum()
        verdad = int((acumulada >= 0.5).idxmax())
        resultado[nombre] = (verdad, acumulada)
        # lecturas de dos errores típicos: acumular en orden de frecuencia (value_counts sin
        # sort=False) y, en la calificación, sumar al total los pedidos sin calificar
        por_frecuencia = int((datos.value_counts(normalize=True).cumsum() >= 0.5).idxmax())
        sobre_todos = int((original[columna].value_counts(normalize=True, dropna=False)
                           .reindex(categorias, fill_value=0).cumsum() >= 0.5).idxmax())
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        if isinstance(x, (pd.Series, pd.DataFrame)) and len(x) == len(categorias):
            print(f"✗  {nombre}: esa es la tabla completa, un valor por categoría; escribe solo la "
                  "categoría en la que el % acumulado llega al 50 %")
            continue
        v, problema = _un_numero(x, n)
        if problema:
            print(f"✗  {nombre}: {problema}")
            continue
        if _cerca(v, verdad, 1e-9):
            print(f"✓  {nombre}")
            bien += 1
        elif _cerca(v, datos.mean(), 0.051):
            print(f"✗  {nombre}: eso es la media de los números; en una ordinal no sabemos si las "
                  "categorías están a la misma distancia: usa la acumulada")
        elif v not in categorias:
            print(f"✗  {nombre}: la mediana de una ordinal es una de sus categorías "
                  f"({categorias[0]} a {categorias[-1]})")
        elif nombre == "calificacion" and v == sobre_todos and v == por_frecuencia:
            print("✗  calificacion: revisa dos cosas: que la acumulada vaya en el orden de las "
                  "categorías, de 1 a 5 (value_counts las ordena por frecuencia si no le dices "
                  "sort=False), y que el total sean los pedidos calificados: los que no tienen "
                  "calificación no cuentan")
        elif nombre == "calificacion" and v == sobre_todos:
            print("✗  calificacion: ¿sobre qué total calculaste la acumulada? La mediana es la de "
                  "los pedidos calificados: los que no tienen calificación no cuentan")
        else:
            orden = (" (con las categorías en su orden, de la menor a la mayor: value_counts las "
                     "ordena por frecuencia si no le dices sort=False)" if v == por_frecuencia else "")
            if acumulada[int(v)] < 0.5:
                print(f"✗  {nombre}: en esa categoría el % acumulado todavía no llega al 50 %{orden}")
            else:
                print(f"✗  {nombre}: en esa categoría la acumulada ya pasó del 50 %, pero no es la "
                      f"primera que llega: mira las anteriores{orden}")

    if bien == 2:
        filas = {}
        for nombre, (verdad, acumulada) in resultado.items():
            posicion = acumulada.index.get_loc(verdad)
            anterior = acumulada.iloc[posicion - 1] if posicion > 0 else 0.0
            filas[nombre] = {"mediana": verdad,
                             "% acumulado en la anterior": anterior * 100,
                             "% acumulado en la mediana": acumulada[verdad] * 100}
        _mostrar(pd.DataFrame(filas).T.round(1).astype({"mediana": int}))
        if all(original[c].dropna().mode()[0] == v for c, (v, _) in resultado.items()):
            print("·  Aquí la mediana coincide con la moda; la mediana sale de la acumulada (la "
                  "primera categoría que llega al 50 %), no de la barra más alta")
        print("✓  Las dos cuadran. Responde abajo: ¿por qué no resumirlas con la media?")


def revisar_por_ciudad(media=..., desviacion=..., ric=..., *sobran, **otros):
    """Ejercicio 3: media, desviación estándar y RIC del tiempo de entrega, por ciudad."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if _sobrantes(sobran, otros, ["media", "desviacion", "ric"]):
        return
    original = _original()
    if original is None:
        return
    n = len(original)
    grupos = original.groupby("ciudad")["tiempo_entrega_min"]
    ciudades = list(grupos.size().index)
    q1, q3 = grupos.quantile(0.25), grupos.quantile(0.75)
    candidatos = {
        "media": grupos.mean(), "mediana": grupos.median(),
        "desviacion": grupos.std(), "desviacion0": grupos.std(ddof=0),
        "varianza": grupos.var(), "varianza0": grupos.var(ddof=0),
        "ric": q3 - q1, "ric_negativo": q1 - q3, "q1": q1, "q3": q3,
        "rango": grupos.max() - grupos.min(),
        "media_total": pd.Series(original["tiempo_entrega_min"].mean(), index=ciudades),
    }
    pocos = int(grupos.size().min())

    def igual(valores, clave, tolerancia=0.051):
        return bool((valores - candidatos[clave]).abs().max() <= tolerancia)

    def leer(nombre, x):
        """Los valores por ciudad, en el orden de `ciudades`, o None (con aviso)."""
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            return None
        if isinstance(x, pd.DataFrame) and "ciudad" in x.columns:
            print(f"✗  {nombre}: la ciudad quedó como columna (¿usaste reset_index?); déjala en "
                  "el índice, como sale de groupby")
            return None
        if isinstance(x, pd.DataFrame) and x.shape[1] > 1:
            columnas = set(map(str, x.columns))
            if columnas & _COLUMNAS:
                print(f"✗  {nombre}: es una tabla con {x.shape[1]} columnas; hace falta una serie con "
                      "un valor por ciudad: ¿elegiste la columna tiempo_entrega_min antes de calcular?")
            elif columnas & set(ciudades):
                print(f"✗  {nombre}: las ciudades quedaron como columnas; déjalas en el índice, como "
                      "sale de groupby")
            else:
                print(f"✗  {nombre}: es una tabla con {x.shape[1]} columnas, una por medida; hace "
                      "falta una serie: quédate solo con la columna de la medida que se pide")
            return None
        s = _serie(x)
        if s is None:
            if isinstance(x, (numbers.Real, np.number)) and not isinstance(x, bool):
                print(f"✗  {nombre}: es un solo número; hace falta un valor por ciudad: agrupa "
                      "por ciudad, como en la celda [dos_ciudades]")
            elif isinstance(x, (list, tuple, np.ndarray)):
                print(f"✗  {nombre}: es una lista sin los nombres de las ciudades; deja el "
                      "resultado de groupby, que trae la ciudad en el índice")
            elif isinstance(x, str):
                print(f"✗  {nombre}: es texto; hace falta una serie de pandas con un valor por ciudad")
            elif x is None:
                print(f"✗  {nombre}: está vacío (None): ¿le asignaste el resultado de un print?")
            elif callable(x):
                print(f"✗  {nombre}: es una función, no su resultado: ¿faltaron los paréntesis "
                      "del final, ()?")
            elif type(x).__name__.endswith("GroupBy"):
                print(f"✗  {nombre}: es el agrupamiento, todavía sin la medida: falta calcularla "
                      "en cada grupo")
            else:
                print(f"✗  {nombre}: tiene que ser una serie de pandas con un valor por ciudad "
                      f"(es {type(x).__name__})")
            return None
        if len(s) == n:
            print(f"✗  {nombre}: esa es la columna entera, un valor por pedido: falta agruparla "
                  "por ciudad y resumirla")
            return None
        if isinstance(s.index, pd.MultiIndex) and s.index.nlevels > 1:
            ultimo = s.index.get_level_values(-1)
            if pd.api.types.is_float_dtype(ultimo) and bool(((ultimo >= 0) & (ultimo <= 1)).all()):
                print(f"✗  {nombre}: su índice tiene {s.index.nlevels} niveles: con una lista entre "
                      "corchetes, como quantile([0.25]), pandas añade un nivel; usa quantile(0.25)")
            else:
                print(f"✗  {nombre}: su índice tiene {s.index.nlevels} niveles: ¿agrupaste por más "
                      "de una columna? Agrupa solo por ciudad")
            return None
        claves = [str(k) for k in s.index]
        if sorted(claves) != sorted(ciudades) or len(set(claves)) != len(claves):
            if isinstance(s.index, pd.RangeIndex):
                print(f"✗  {nombre}: sus filas van numeradas, sin el nombre de la ciudad; deja el "
                      "resultado de groupby, que trae la ciudad en el índice")
            elif set(claves) < set(ciudades):
                faltan = ", ".join(c for c in ciudades if c not in claves)
                print(f"✗  {nombre}: le faltan ciudades ({faltan}): ¿partiste de dos, la tabla de "
                      "la celda [dos_ciudades]? Agrupa pedidos, con todas las ciudades")
            else:
                print(f"✗  {nombre}: sus filas tienen que ser las cinco ciudades: ¿agrupaste por "
                      "ciudad?")
            return None
        if pd.api.types.is_bool_dtype(s):
            print(f"✗  {nombre}: sus valores son True o False; hace falta un número por ciudad")
            return None
        if s.dtype == object and all(isinstance(v, numbers.Real) and not isinstance(v, bool)
                                     for v in s):
            s = s.astype(float)   # números guardados como object
        if (not pd.api.types.is_numeric_dtype(s)) or pd.api.types.is_complex_dtype(s):
            print(f"✗  {nombre}: sus valores son texto; déjalos como números (sin astype(str))")
            return None
        valores = pd.Series(s.to_numpy(dtype=float), index=claves).reindex(ciudades)
        if valores.isna().any():
            print(f"✗  {nombre}: tiene valores nan: si restaste dos series, revisa que las dos "
                  "tengan las mismas ciudades en el índice")
            return None
        return valores

    bien = 0
    valores = leer("media", media)
    if valores is not None:
        if igual(valores, "media"):
            print("✓  media")
            bien += 1
        elif igual(valores, "mediana"):
            print("✗  media: esas son las medianas; la media suma y divide")
        elif igual(valores, "media_total"):
            print("✗  media: es la misma media para todas las ciudades: falta agrupar")
        elif igual(valores, "media", 0.51):
            print("✗  media: casi: perdiste precisión al redondear; deja al menos dos decimales")
        elif bool((valores == valores.round()).all()) and igual(valores, "media", 1.0):
            print("✗  media: le quitaste los decimales (¿astype(int)?); deja al menos dos decimales")
        else:
            print("✗  media: no cuadra: ¿la calculaste sobre tiempo_entrega_min?")

    valores = leer("desviacion", desviacion)
    if valores is not None:
        if igual(valores, "desviacion") or (pocos > 1000 and igual(valores, "desviacion0")):
            print("✓  desviacion")
            bien += 1
        elif (bool(((valores / candidatos["varianza"] - 1).abs() <= 0.01).all())
              or bool(((valores / candidatos["varianza0"] - 1).abs() <= 0.01).all())):
            print("✗  desviacion: eso es la varianza, en minutos al cuadrado; la desviación "
                  "vuelve a minutos")
        elif igual(valores, "ric"):
            print("✗  desviacion: eso parece el RIC; la desviación mide la distancia a la media")
        elif igual(valores, "rango"):
            print("✗  desviacion: eso es el rango (máximo menos mínimo)")
        elif igual(valores, "media"):
            print("✗  desviacion: esas son las medias")
        elif igual(valores, "desviacion", 0.51):
            print("✗  desviacion: casi: perdiste precisión al redondear; deja al menos dos decimales")
        else:
            print("✗  desviacion: no cuadra: revisa la columna y la función que usaste")

    valores = leer("ric", ric)
    if valores is not None:
        if igual(valores, "ric"):
            print("✓  ric")
            bien += 1
        elif igual(valores, "ric_negativo"):
            print("✗  ric: salió negativo: revisa el orden de la resta")
        elif igual(valores, "q3") or igual(valores, "q1"):
            print("✗  ric: eso es un cuartil; el RIC es la distancia entre dos cuartiles")
        elif igual(valores, "rango"):
            print("✗  ric: eso es el rango (máximo menos mínimo); el RIC usa los cuartiles")
        elif igual(valores, "mediana"):
            print("✗  ric: esas son las medianas")
        elif igual(valores, "desviacion") or igual(valores, "desviacion0"):
            print("✗  ric: esa es la desviación estándar")
        elif igual(valores, "ric", 0.51):
            print("✗  ric: casi: perdiste precisión al redondear; deja al menos dos decimales")
        else:
            print("✗  ric: no cuadra: ¿qué cuantiles restaste? (en quantile, 0.25 es el 25 %)")

    if bien == 3:
        tabla = pd.DataFrame({"media": candidatos["media"], "desviación": candidatos["desviacion"],
                              "RIC": candidatos["ric"]})
        _mostrar(tabla.round(1))
        print("✓  Las tres cuadran. Responde abajo: ¿en qué ciudad se puede prometer una ventana "
              "más estrecha?")


def revisar_cuartiles(*sueltos, q1=..., q3=..., ric=..., **otros):
    """Ejercicio 4: Q1, Q3 y RIC de distancia_km, recibidos por nombre."""
    import numpy as np
    import pandas as pd

    if sueltos:
        print("✗  Pasa los valores por nombre, como en la plantilla: "
              "revisar_cuartiles(q1=q1, q3=q3, ric=ric)")
        return
    if _sobrantes((), otros, ["q1", "q3", "ric"]):
        return
    original = _original()
    if original is None:
        return
    d = original["distancia_km"]
    n = len(d)
    verdad = {"q1": d.quantile(0.25), "q3": d.quantile(0.75)}
    verdad["ric"] = verdad["q3"] - verdad["q1"]
    otros_valores = {"mediana": d.median(), "media": d.mean(), "mínimo": d.min(),
                     "máximo": d.max(), "desviación": d.std()}
    # los mismos cálculos sobre otras columnas: la celda [cuartiles] usa tiempo_entrega_min
    otras = original.select_dtypes("number").drop(columns="distancia_km")
    de_otras = {"q1": list(otras.quantile(0.25)), "q3": list(otras.quantile(0.75)),
                "ric": list(otras.quantile(0.75) - otras.quantile(0.25))}
    tol = 0.051
    bien = 0

    for nombre, x in [("q1", q1), ("q3", q3), ("ric", ric)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        if nombre == "ric" and isinstance(x, np.generic) and x is q1:   # ric = q1
            print("✗  ric: es el mismo valor que pasaste en q1; el RIC es la distancia entre los "
                  "dos cuartiles")
            continue
        v, problema = _un_numero(x, n)
        if problema:
            print(f"✗  {nombre}: {problema}")
            continue
        if _cerca(v, verdad[nombre], tol):
            print(f"✓  {nombre}")
            bien += 1
            continue
        if v != round(v) and any(_cerca(v, o, 1e-6) for o in de_otras[nombre]):
            print(f"✗  {nombre}: ese valor es de otra columna: ¿lo calculaste sobre distancia_km?")
            continue
        if nombre == "ric":
            if _cerca(v, -verdad["ric"], tol):
                print("✗  ric: salió negativo: revisa el orden de la resta")
            elif _cerca(v, otros_valores["máximo"] - otros_valores["mínimo"], tol):
                print("✗  ric: eso es el rango (máximo menos mínimo); el RIC usa los cuartiles")
            elif _cerca(v, otros_valores["desviación"], tol):
                print("✗  ric: esa es la desviación estándar")
            else:
                print("✗  ric: no cuadra: es la distancia entre el tercer y el primer cuartil")
            continue
        otro = "q3" if nombre == "q1" else "q1"
        if _cerca(v, verdad[otro], tol):
            print(f"✗  {nombre}: ese es el otro cuartil: ¿los intercambiaste?")
        elif any(_cerca(v, valor, tol) for clave, valor in otros_valores.items()):
            clave = next(c for c, valor in otros_valores.items() if _cerca(v, valor, tol))
            print(f"✗  {nombre}: eso es {'la' if clave in ('mediana', 'media', 'desviación') else 'el'} "
                  f"{clave}")
        else:
            parte = "25 %" if nombre == "q1" else "75 %"
            print(f"✗  {nombre}: no cuadra: es el valor que deja por debajo el {parte} de los pedidos")

    if bien == 3:
        tabla = pd.DataFrame({"km": [verdad["q1"], d.median(), verdad["q3"], verdad["ric"]]},
                             index=["Q1", "mediana", "Q3", "RIC"])
        _mostrar(tabla.round(2))
        print("✓  Los tres cuadran. Ahora dibuja el diagrama de caja (el verificador no lo revisa) "
              "y compáralo con estos cuartiles")

# Crea datos/pedidos.csv: los mismos 12 000 pedidos de las sesiones 1 y 2
ruta = crear_datos("datos")

## 1 · Los datos, con sus escalas

Cargamos los pedidos y declaramos las escalas, como en las sesiones 1 y 2: la escala de cada
columna decide qué medida de centro tiene sentido.

In [ ]:
# [cargar]
import pandas as pd
import matplotlib.pyplot as plt

pedidos = pd.read_csv("datos/pedidos.csv")
# Las escalas que declaramos en la sesión 1
franjas = ["Mañana", "Almuerzo", "Tarde", "Noche"]
pedidos["franja"] = pd.Categorical(
    pedidos["franja"], categories=franjas, ordered=True)
pedidos["estrato"] = pd.Categorical(
    pedidos["estrato"], categories=[1, 2, 3, 4, 5, 6], ordered=True)
pedidos["calificacion"] = pd.Categorical(
    pedidos["calificacion"], categories=[1, 2, 3, 4, 5], ordered=True)
pedidos.shape

### Diccionario de datos (el mismo de la sesión 1)

| Columna | Qué registra | Valores o unidad |
|---|---|---|
| `id_pedido` | Código del pedido | P00001, P00002… |
| `fecha` | Día del pedido | del 1 de enero al 30 de junio de 2026 |
| `franja` | Momento del día en que se pidió | Mañana, Almuerzo, Tarde, Noche |
| `ciudad` | Ciudad de entrega | Bogotá, Medellín, Cali, Barranquilla, Bucaramanga |
| `categoria` | Tipo de comercio que despacha | Restaurante, Mercado, Farmacia, Mascotas, Floristería |
| `n_productos` | Unidades pedidas | conteo |
| `valor_pedido` | Valor de los productos, sin domicilio ni propina | pesos (COP) |
| `medio_pago` | Cómo se pagó | Efectivo, Tarjeta, Billetera digital |
| `estrato` | Estrato de la **dirección de entrega** (lo asigna el municipio a la vivienda) | 1 a 6 |
| `suscriptor` | Si quien pide paga la suscripción de Tingua | Sí / No |
| `distancia_km` | Distancia por vía del comercio a la dirección | kilómetros; la app registra hasta 14: una ruta más larga queda como 14 |
| `lluvia` | Si llovía en la zona durante la entrega | Sí / No |
| `temperatura_c` | Temperatura en la zona al momento de la entrega | grados Celsius |
| `tiempo_entrega_min` | Minutos desde que se confirma el pedido hasta que llega | minutos |
| `calificacion` | Estrellas que dio quien pidió; vacía si no calificó | 1 a 5 |

## 2 · El centro: ¿cuánto tarda un pedido típico?

### La media y la mediana

La **media** suma los valores y divide entre cuántos son: $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$.
Es el punto de equilibrio de los datos: las distancias a la media de los que quedan por encima
compensan las de los que quedan por debajo. La **mediana** es el valor del medio cuando se ordenan
los datos: deja la mitad de los pedidos a cada lado.

In [ ]:
# [centro]
tiempo = pedidos["tiempo_entrega_min"]
print(f"Media:   {tiempo.mean():.1f} min")
print(f"Mediana: {tiempo.median():.1f} min")

### La moda de una continua es una clase, no un valor

La **moda** es el valor más frecuente. En una variable continua casi ningún valor se repite, y el
que más se repite lo decide el azar del redondeo. Lo que sí tiene sentido es la **clase modal**: el
intervalo con más pedidos.

In [ ]:
# [moda]
# La moda de los valores crudos: un tiempo que se repite por azar
moda = tiempo.mode()[0]
print("Moda de los valores:", moda, "min,",
      "en", (tiempo == moda).sum(), "pedidos")
# La clase modal: el intervalo de 5 min con más pedidos
clases = pd.cut(tiempo, bins=range(0, 210, 5), right=False)
conteo = clases.value_counts(sort=False)
print("Clase modal:", conteo.idxmax(), "con", conteo.max(), "pedidos")

### El histograma, con la media y la mediana

Es el histograma de la sesión 2 con dos líneas: la media y la mediana. La cola a la derecha
separa una de la otra.

In [ ]:
# [histograma_centro]
media, mediana = tiempo.mean(), tiempo.median()
ax = tiempo.plot.hist(bins=range(0, 210, 5), color="#8A2E9E", alpha=0.45,
                      edgecolor="white", label="Pedidos")
# Media en naranja, mediana en morado oscuro y punteada
ax.axvline(media, color="#F46201", linewidth=2.5,
           label=f"Media: {media:.1f} min")
ax.axvline(mediana, color="#3A0F47", linewidth=2.5, linestyle="--",
           label=f"Mediana: {mediana:.1f} min")
ax.set_xlim(0, 100)   # la cola sigue hasta 201 min
ax.legend()
ax.set_title("Tiempo de entrega, con su media y su mediana")
ax.set_xlabel("Minutos")
ax.set_ylabel("Pedidos")
plt.show()

### Media frente a mediana

Con los cinco tiempos del diagnóstico de la sesión 1 se ve por qué se separan: un solo pedido muy
lento arrastra la media; la mediana solo mira el orden.

In [ ]:
# [cinco]
cinco = pd.Series([20, 22, 25, 27, 150])   # los tiempos del diagnóstico
print("Media:", cinco.mean(), "· mediana:", cinco.median())
# ¿Y si el pedido de 150 min hubiera tardado 300?
cinco[4] = 300
print("Media:", cinco.mean(), "· mediana:", cinco.median())

In [ ]:
# [valor]
# El valor del pedido: una cola a la derecha mucho más larga
valor = pedidos["valor_pedido"]
print(f"Media: {valor.mean():.0f} pesos · mediana: {valor.median():.0f} pesos")

### ¿Qué centro admite cada tipo de variable?

| Tipo de variable | Medida de centro |
|---|---|
| Nominal | La moda |
| Ordinal | La mediana (y la moda) |
| Cuantitativa | La media o la mediana, según la forma; en una continua, la clase modal |

La mediana de una ordinal sí tiene sentido: solo necesita el orden. Pero pandas no la calcula para
una categoría ordenada: da `TypeError`. Es lo contrario de la sesión 1, donde se negaba a promediar
el estrato porque eso no tenía sentido. Aquí el error es una limitación de pandas, y la mediana se
lee en la acumulada.

In [ ]:
# [mediana_ordinal]
# La mediana de una ordinal tiene sentido, pero pandas no la calcula
try:
    pedidos["franja"].median()
except TypeError as error:
    print("TypeError:", error)

In [ ]:
# [acumulada_franja]
# Se lee en la acumulada: la primera franja que llega al 50 %
franja = pedidos["franja"].value_counts(sort=False, normalize=True)
(franja.cumsum() * 100).round(1)

### Para profundizar: la media de datos agrupados

Si solo tienes la tabla de intervalos, y no los 12 000 tiempos, puedes aproximar la media con la
**marca de clase** $m_i$, el punto medio de cada intervalo, que representa a los $n_i$ pedidos que
caen en él:

$$\bar{x} \approx \frac{1}{n}\sum_{i} n_i\, m_i$$

Con intervalos de 5 minutos, la aproximación queda muy cerca de la media verdadera. La tabla de
10 minutos de la sesión 2 no sirve para esto: su última clase, [80, inf), es abierta y no tiene
punto medio.

In [ ]:
# [agrupada]
# Para profundizar: la media aproximada con las marcas de clase
clases = pd.cut(tiempo, bins=range(0, 210, 5), right=False)
conteo = clases.value_counts(sort=False)
marcas = [intervalo.mid for intervalo in conteo.index]   # puntos medios
agrupada = (conteo * marcas).sum() / conteo.sum()
print(f"Media con las marcas de clase: {agrupada:.2f} min")
print(f"Media con los 12 000 tiempos:  {tiempo.mean():.2f} min")

## 3 · La dispersión: ¿qué tan variable es?

### El rango

El **rango** es el máximo menos el mínimo. Es fácil de calcular, pero solo mira dos pedidos: los
dos más extremos.

In [ ]:
# [rango]
print("Mínimo:", tiempo.min(), "· máximo:", tiempo.max())
print(f"Rango: {tiempo.max() - tiempo.min():.1f} min")

### Varianza y desviación estándar

La **varianza** promedia los cuadrados de las distancias a la media; la **desviación estándar** es
su raíz, y vuelve a quedar en minutos:

$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N} (x_i - \mu)^2 \qquad s^2 = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})^2$$

$\sigma$ (divide entre $N$) es la de una población: aquí, los 12 000 pedidos. $s$ (divide entre
$n - 1$) es la de una muestra. En pandas, `std()` divide entre $n - 1$; con `ddof=0`, entre $N$.

In [ ]:
# [desviacion]
print(f"Varianza:       {tiempo.var():.1f} min²")
print(f"s, entre n - 1: {tiempo.std():.3f} min")      # lo que hace std()
print(f"σ, entre N:     {tiempo.std(ddof=0):.3f} min")

### La trampa: pandas y numpy no dividen igual

Con muchos datos no se nota; con pocos, sí. `ddof` dice cuánto se le resta a $n$ al dividir.

In [ ]:
# [ddof]
import numpy as np

cinco = [20, 22, 25, 27, 150]   # los tiempos del diagnóstico
print(f"pandas: {pd.Series(cinco).std():.2f}")   # divide entre n - 1
print(f"numpy:  {np.std(cinco):.2f}")            # divide entre n
print(f"pandas con ddof=0: {pd.Series(cinco).std(ddof=0):.2f}")

### La desviación, en minutos

¿Qué parte de los pedidos queda a menos de una desviación de la media?

In [ ]:
# [dentro]
media, s = tiempo.mean(), tiempo.std()
# between: True si el tiempo está entre los dos valores
cerca = tiempo.between(media - s, media + s)
# La media de una columna de True y False es la proporción de True
print(f"De {media - s:.1f} a {media + s:.1f} min:",
      f"{cerca.mean() * 100:.1f} % de los pedidos")

### Cuartiles, rango intercuartílico y `describe()`

Los **cuartiles** parten los datos ordenados en cuatro partes con la misma cantidad de pedidos:
por debajo de $Q_1$ queda el 25 %; de $Q_2$ (la mediana), el 50 %, y de $Q_3$, el 75 %. El **rango
intercuartílico** (RIC) es $Q_3 - Q_1$: el ancho del 50 % central. `describe()` da el resumen de
cinco números (mínimo, $Q_1$, mediana, $Q_3$ y máximo), con la media y la desviación.

In [ ]:
# [cuartiles]
q1 = tiempo.quantile(0.25)
q3 = tiempo.quantile(0.75)
print(f"Q1 = {q1:.1f} · Q3 = {q3:.1f} · RIC = {q3 - q1:.1f} min")
tiempo.describe().round(1)

### El diagrama de caja, por ciudad

La caja va de $Q_1$ a $Q_3$, con la mediana adentro; los bigotes llegan hasta el último pedido que
no se aleja más de 1,5 RIC de la caja, y los puntos sueltos son los que se alejan más:
**candidatos a atípicos**, el tema de la sesión 4.

In [ ]:
# [cajas]
# groupby("ciudad") separa los pedidos de cada ciudad
por_ciudad = pedidos.groupby("ciudad")["tiempo_entrega_min"]
orden = por_ciudad.median().sort_values().index   # de menor a mayor mediana
tiempos = [por_ciudad.get_group(ciudad) for ciudad in orden]

fig, ax = plt.subplots()
ax.boxplot(tiempos, widths=0.5, patch_artist=True,
           boxprops={"facecolor": "#EFE3F2", "edgecolor": "#8A2E9E"},
           medianprops={"color": "#3A0F47", "linewidth": 2.5},
           whiskerprops={"color": "#8A2E9E"},
           capprops={"color": "#8A2E9E"},
           flierprops={"marker": ".", "markersize": 4,
                       "markerfacecolor": "#8A2E9E",
                       "markeredgecolor": "none"})
ax.set_xticks(range(1, len(orden) + 1), orden)
ax.set_title("Tiempo de entrega por ciudad")
ax.set_ylabel("Minutos")
plt.show()

### ¿Qué medidas se mueven con la cola?

Una medida es **robusta** si unos pocos datos extremos no la mueven. Quitemos el 0,5 % de pedidos
más lentos y comparemos.

In [ ]:
# [robustez]
# Sin el 0,5 % más lento: los pedidos por encima del percentil 99,5
corte = tiempo.quantile(0.995)
sin_cola = tiempo[tiempo <= corte]
print(f"Salen {len(tiempo) - len(sin_cola)} pedidos,",
      f"de más de {corte:.1f} min")
print(f"Media:      {tiempo.mean():.1f} → {sin_cola.mean():.1f}")
print(f"Desviación: {tiempo.std():.1f} → {sin_cola.std():.1f}")
print(f"Mediana:    {tiempo.median():.1f} → {sin_cola.median():.1f}")

### Dos ciudades con casi la misma media

`groupby` separa los pedidos de cada ciudad y calcula lo mismo en cada grupo.

In [ ]:
# [dos_ciudades]
dos = pedidos[pedidos["ciudad"].isin(["Barranquilla", "Bucaramanga"])]
grupos = dos.groupby("ciudad")["tiempo_entrega_min"]
resumen = pd.DataFrame({
    "media": grupos.mean(),
    "desviación": grupos.std(),
    "RIC": grupos.quantile(0.75) - grupos.quantile(0.25),
})
resumen.round(1)

### El coeficiente de variación

El **coeficiente de variación** (CV) es la desviación dividida entre la media: compara la
variabilidad de cantidades con unidades o tamaños distintos. Solo vale para variables de **razón**
(con un cero verdadero) y media positiva; nunca para la temperatura, que es de intervalo.

In [ ]:
# [cv]
# Coeficiente de variación: la desviación como fracción de la media
valor = pedidos["valor_pedido"]
print(f"Tiempo: CV = {tiempo.std() / tiempo.mean():.2f}")
print(f"Valor:  CV = {valor.std() / valor.mean():.2f}")

---
# Práctica guiada

Cuatro ejercicios. Cada celda termina con un verificador que dice qué está bien y, en lo demás, te
da una pista sin darte la respuesta. Reemplaza cada `...` y ejecuta; puedes repetir la celda las
veces que quieras.

## Ejercicio 1 · El centro de `n_productos`

Calcula la media, la mediana y la moda de `n_productos`. Es discreta y sus valores se repiten
mucho: aquí `mode()` sí tiene sentido. Pista: las celdas [centro] y [moda].

Después responde abajo, en «Tus respuestas»: ¿cuál de las tres representa mejor al pedido típico?
¿Por qué?

In [ ]:
# [ej1]
media = ...      # media de n_productos
mediana = ...    # mediana de n_productos
moda = ...       # moda de n_productos
revisar_centro(media=media, mediana=mediana, moda=moda)

## Ejercicio 2 · La mediana de dos ordinales

El `estrato` y la `calificacion` son ordinales, y `median()` no funciona con ellas (celda
[mediana_ordinal]). Calcula el % acumulado de cada una, como en la celda [acumulada_franja], y
escribe su mediana: la **primera** categoría en la que el % acumulado llega al 50 %. La de la
calificación es la de los pedidos **calificados**.

Después responde abajo: ¿por qué no resumir estas dos columnas con la media?

In [ ]:
# [ej2]
# Calcula aquí el % acumulado del estrato y el de la calificación


# Escribe la mediana que leíste en cada acumulada
mediana_estrato = ...
mediana_calificacion = ...
revisar_medianas(estrato=mediana_estrato, calificacion=mediana_calificacion)

## Ejercicio 3 · El tiempo de entrega, ciudad por ciudad

Calcula, para cada una de las cinco ciudades, la media, la desviación estándar y el RIC del tiempo
de entrega: tres series con un valor por ciudad. Pista: la celda [dos_ciudades], con todas las
ciudades.

Después responde abajo: ¿en qué ciudad se le puede prometer a la clientela una ventana de entrega
más estrecha? ¿Por qué no basta con mirar la media?

In [ ]:
# [ej3]
media = ...        # media del tiempo de entrega, por ciudad
desviacion = ...   # desviación estándar, por ciudad
ric = ...          # Q3 - Q1, por ciudad
revisar_por_ciudad(media=media, desviacion=desviacion, ric=ric)

## Ejercicio 4 · Los cuartiles de la distancia

Calcula $Q_1$, $Q_3$ y el RIC de `distancia_km`. Deja la última línea como está: entrega cada valor
con su nombre (`q1=`, `q3=`, `ric=`) para que el verificador no los confunda. Pista: la celda
[cuartiles].

In [ ]:
# [ej4]
q1 = ...    # primer cuartil de distancia_km
q3 = ...    # tercer cuartil de distancia_km
ric = ...   # rango intercuartílico
revisar_cuartiles(q1=q1, q3=q3, ric=ric)

Ahora dibuja el diagrama de caja de `distancia_km`, con una sola caja. Pista: la celda [cajas]; con
una sola columna, `ax.boxplot` la recibe directamente, sin `groupby`.
**El verificador no revisa el diagrama**: compáralo tú con tus cuartiles. ¿Qué ves por encima del
bigote superior? Pista: el diccionario de datos.

In [ ]:
# [ej4_caja]
# Dibuja aquí el diagrama de caja de distancia_km

### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: al pedido típico lo representa mejor la …, porque …
2. Ejercicio 2: no resumo el estrato ni la calificación con la media porque …
3. Ejercicio 3: la ventana más estrecha se puede prometer en …, porque …; la media no basta porque …
4. Ejercicio 4: por encima del bigote superior veo …, y creo que es porque …